# hashing.ipynb
## Creator: kccaterworld
## Contributors: kccaterworld
### Status: Unfinished, Stable, Innacurate

In [1]:
def checkBin(bins, length=32):
    for number in bins:
        if len(number) != length:
            raise ValueError(f"Binary string must be {length} bits long")
        if not all(bit in '01' for bit in number):
            raise ValueError("Input must be a binary string")
        if type(number) != str:
            raise TypeError("Input must be a string")

def binRot(bin, n):
    checkBin((bin,))
    if not (0 <= n <= 32):
        raise ValueError("Rotation amount must be between 0 and 32")
    return bin[n:] + bin[:n]

def binPadder(bin, length):
    if length < len(bin):
        bin = bin[len(bin)-length:]
    while len(bin) < length:
        bin = "0" + bin
    return bin

def bNot(bin):
    result = ""
    for digit in bin:
        result += f"{"1" if digit == "0" else "0"}"
        continue
    return result

def bAdd(binA, binB):
    intA = int(binA, 2)
    intB = int(binB, 2)
    return binPadder(bin((intA + intB) % (2**32))[2:], 32)

In [2]:
def bAnd(binA, binB):
    result = ""
    for i in range(len(binA)):
        if binA[i] == "0" or binB[i] == "0":
            result += "0"
            continue
        if binA[i] == "1" and binB[i] == "1":
            result += "1"
            continue
    return result

def bOr(binA, binB):
    result = ""
    for i in range(len(binA)):
        if binA[i] == "1" or binB[i] == "1":
            result += "1"
            continue
        if binA[i] == "0" and binB[i] == "0":
            result += "0"
            continue
    return result

def bXor(binA, binB):
    result = ""
    for i in range(len(binA)):
        if binA[i] != binB[i]:
            result += "1"
            continue
        if binA[i] == binB[i]:
            result += "0"
            continue
    return result

In [3]:
def concat(lists):
    result = []
    for list in lists:
        result += list
    return result

def prettyPrint(text):
    returned = ""
    for i in range(len(text)):
        if (i % 64 == 0):
            returned += "\n"
        if (i % 8 == 0):
            returned += " "
        returned += text[i]
    return returned[1:]

def encode(text):
    encoded = ""
    for char in text:
        encoded += char.encode("utf-8").hex()
    return bin(int(encoded, 16))[2:]

def sizeList(metalist):
    return (len(metalist), round(sum(len(row) for row in metalist) / len(metalist)))

def preprep(text):
    if text == "":
        return "".join(["0" for i in range(512)])
    bintext = encode(text)
    padded = bintext + "1"
    while len(padded) % 448 != 0:
        padded += "0"
    padded += binPadder(bin(len(bintext))[2:], 64)
    while len(padded) % 512 != 0:
        padded += "0"
    return padded

In [4]:
def binHex(bin):
    hexnUM = ""
    for i in range(0, len(bin), 4):
        hexnUM += hex(int(bin[i:i+4], 2))[2:]
    return hexnUM

print(binHex("1101"))

d


In [5]:
def F(b, c, d, i):
    if 0 <= i <= 15:
        return  bOr(bAnd(b, c), bAnd(bNot(b), d))
    if 16 <= i <= 31:
        return bOr(bAnd(d, b), bAnd(bNot(d), c))
    if 32 <= i <= 47:
        return bXor(bXor(b, c), d)
    if 48 <= i <= 63:
        return bXor(c, bAnd(b, bNot(d)))

def combine(a, b, c, d, input, i):
    return bAdd(binRot(bAdd(bAdd(bAdd(F(b, c, d, i), a), input), K[i]), int(r[i])), b)

In [6]:
a = str(binPadder(bin(0x01234567)[2:], 32))
b = str(binPadder(bin(0x89abcdef)[2:], 32))
c = str(binPadder(bin(0xfedcba98)[2:], 32))
d = str(binPadder(bin(0x76543210)[2:], 32))
len(a), len(b), len(c), len(d)
a, b, c, d

('00000001001000110100010101100111',
 '10001001101010111100110111101111',
 '11111110110111001011101010011000',
 '01110110010101000011001000010000')

In [7]:
chunkList = [preprep("bleep")[i:i + 32] for i in range(0, len(preprep("bleep")), 32)]
sizeList(chunkList)

(16, 32)

In [8]:
shiftrot = [[kval.strip() for kval in line.split(",")] for line in open("shiftrot.txt", "r").read().split("\n")]
K = concat(shiftrot[17:33])
r = concat(shiftrot[34:42])

In [9]:
binTestA =   "00110110111111000011110100100010"
binTestB =   "00001101111001000101000001101010"
bintTestA = 0b00110110111111000011110100100010
bintTestB = 0b00001101111001000101000001101010

In [10]:
# Tests of bitwise operations
print(bAnd(binTestA, binTestB) == binPadder(str(bin(bintTestA & bintTestB)[2:]), 32))
print(bOr(binTestA, binTestB) == binPadder(str(bin(bintTestA | bintTestB)[2:]), 32))
print(bXor(binTestA, binTestB) == binPadder(str(bin(bintTestA ^ bintTestB)[2:]), 32))
print(bNot(binTestA) != False) #Manually verified because the built-in bitwise NOT operator didn't work
print(bAdd(binTestA, binTestB) == binPadder(str(bin((bintTestA + bintTestB) % (2**32))[2:]), 32))
print(binRot(binTestA, 5) == binPadder(str(bin(((bintTestA << 5) | (bintTestA >> (32 - 5))) % (2**32))[2:]), 32))

True
True
True
True
True
True


In [11]:

def md5(plaintext):
    prepped = preprep(plaintext)
    if len(prepped) != 512:
        raise ValueError("Prepped text is not 512 bits")
    chunks = [prepped[i:i + 32] for i in range(0, len(prepped), 32)]
    if sizeList(chunks) != (16, 32):
        raise ValueError("Chunks got split wrong")
    a = str(binPadder(bin(0x01234567)[2:], 32))
    b = str(binPadder(bin(0x89abcdef)[2:], 32))
    c = str(binPadder(bin(0xfedcba98)[2:], 32))
    d = str(binPadder(bin(0x76543210)[2:], 32))
    for i in range(64):
        if 0 <= i <= 15:
            cP, dP, aP = b, c, d
            bP = combine(a, b, c, d, chunks[i], i)
            a, b, c, d = aP, bP, cP, dP
            continue
        if 16 <= i <= 31:
            cP, dP, aP = b, c, d
            bP = combine(a, b, c, d, chunks[((5 * i) + 1) % 16], i)
            a, b, c, d = aP, bP, cP, dP
            continue
        if 32 <= i <= 47:
            cP, dP, aP = b, c, d
            bP = combine(a, b, c, d, chunks[((3 * i) + 5) % 16], i)
            a, b, c, d = aP, bP, cP, dP
            continue
        if 48 <= i <= 63:
            cP, dP, aP = b, c, d
            bP = combine(a, b, c, d, chunks[(7 * i) % 16], i)
            a, b, c, d = aP, bP, cP, dP
            continue
    def to_hex_little_endian(word):
        return ''.join(f"{(int(word, 2) >> (8*i)) & 0xff:02x}" for i in range(4))
    digest = ''.join(to_hex_little_endian(x) for x in [a, b, c, d])

    #ciphertext = binHex(a) + binHex(b) + binHex(c) + binHex(d)
    return digest

def sha1(plaintext):
    ciphertext = ""
    return ciphertext

def sha256(plaintext):
    ciphertext = ""
    return ciphertext

In [12]:
import cryptography
import hashlib
phrase = ""
print(f"Phrase: {phrase}")
print(f"My MD5: {md5(phrase)}. Length: {len(md5(phrase))} characters, {len(md5(phrase))*4} bits")
print(f"Hashlib MD5: {hashlib.md5(phrase.encode()).hexdigest()}. Length: {len(hashlib.md5(phrase.encode()).hexdigest())} characters, {len(hashlib.md5(phrase.encode()).hexdigest())*4} bits")
print(f"Comparison of MD5: {md5(phrase) == hashlib.md5(phrase.encode()).hexdigest()}")

Phrase: 
My MD5: d5f24635200be7af73d6ae48f1038a3c. Length: 32 characters, 128 bits
Hashlib MD5: d41d8cd98f00b204e9800998ecf8427e. Length: 32 characters, 128 bits
Comparison of MD5: False
